In [ ]:
# ============================================================
#   Datasets : BDD100K + GTSRB + Road Vehicle Images
#   Tasks    : Detection · Segmentation · Sign Cls · Vehicle Cls
# ============================================================

!pip install ultralytics -q   

import os, json, cv2, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")


# ============================================================
#   STEP 1 — DYNAMIC PATH DISCOVERY
# ============================================================

INPUT_ROOT = "/kaggle/input"


def _dataset_root(slug_fragment, max_depth=6):
    """
    Return the first directory whose basename contains slug_fragment.
    Walks up to max_depth levels so it works for both
      /kaggle/input/<slug>/                      (old layout)
      /kaggle/input/datasets/<user>/<slug>/      (new layout)
    """
    if not os.path.exists(INPUT_ROOT):
        return None
    base_depth = INPUT_ROOT.rstrip("/").count("/")
    for dp, dirs, _ in os.walk(INPUT_ROOT):
        if dp.count("/") - base_depth > max_depth:
            dirs.clear()
            continue
        if slug_fragment.lower() in os.path.basename(dp).lower():
            return dp
    return None


def _find_dir_containing(root, filename_fragment, ext=None, max_walk=7):
    if not root:
        return None
    base_depth = root.rstrip("/").count("/")
    for dp, dirs, files in os.walk(root):
        if dp.count("/") - base_depth > max_walk:
            dirs.clear(); continue
        for f in files:
            p = Path(f)
            if (filename_fragment.lower() in f.lower() and
                    (ext is None or p.suffix.lower() == ext.lower())):
                return dp
    return None


def _find_dir_with_image_subfolders(root, min_classes=2, max_walk=5):
    if not root or not os.path.isdir(root):
        return None
    base_depth = root.rstrip("/").count("/")
    best = None
    for dp, dirs, files in os.walk(root):
        if dp.count("/") - base_depth > max_walk:
            dirs.clear(); continue
        class_dirs = [d for d in dirs
                      if os.path.isdir(os.path.join(dp, d))]
        if len(class_dirs) < min_classes:
            continue
        imgs_found = sum(
            1 for cd in class_dirs[:6]
            for f in os.listdir(os.path.join(dp, cd))
            if Path(f).suffix.lower() in {".jpg", ".png", ".jpeg", ".ppm"}
        )
        if imgs_found >= min(2, len(class_dirs)):
            best = dp
    return best


def _find_split_dir(root, split_name):
    if not root:
        return None
    base_depth = root.rstrip("/").count("/")
    for dp, dirs, files in os.walk(root):
        if dp.count("/") - base_depth > 7:
            dirs.clear(); continue
        if os.path.basename(dp) == split_name:
            if any(f.lower().endswith((".jpg", ".png")) for f in files):
                return dp
    return None


def _find_bdd_seg(bdd_root, kind):
    if not bdd_root:
        return None
    base_depth = bdd_root.rstrip("/").count("/")
    for dp, dirs, files in os.walk(bdd_root):
        if dp.count("/") - base_depth > 8:
            dirs.clear(); continue
        parts = dp.replace("\\", "/").split("/")
        if (kind in parts and "train" in parts
                and ("seg" in parts or "segmentation" in parts)
                and os.listdir(dp)):
            return dp
    return None


# ── Discover all dataset roots ────────────────────────────────
print("\nLocating datasets…")
BDD_ROOT   = _dataset_root("bdd100k")
GTSRB_ROOT = _dataset_root("gtsrb")
VEH_ROOT   = _dataset_root("road-vehicle") or _dataset_root("vehicle")
print(f"  BDD  root : {BDD_ROOT}")
print(f"  GTSRB root: {GTSRB_ROOT}")
print(f"  Veh  root : {VEH_ROOT}")

BDD_IMG_TRAIN = _find_split_dir(BDD_ROOT, "train")
BDD_IMG_VAL   = _find_split_dir(BDD_ROOT, "val")

_det_dir = _find_dir_containing(BDD_ROOT or INPUT_ROOT, "det_train", ext=".json")
BDD_LABEL_JSON = None
if _det_dir:
    for _f in os.listdir(_det_dir):
        if "det_train" in _f and _f.endswith(".json"):
            BDD_LABEL_JSON = os.path.join(_det_dir, _f); break

BDD_SEG_TRAIN = _find_bdd_seg(BDD_ROOT, "images")
BDD_SEG_MASK  = _find_bdd_seg(BDD_ROOT, "labels")
VEHICLE_DIR   = _find_dir_with_image_subfolders(VEH_ROOT, min_classes=2)

print("\n── Path Summary ──")
for _n, _v in [
    ("BDD_IMG_TRAIN",  BDD_IMG_TRAIN),
    ("BDD_IMG_VAL",    BDD_IMG_VAL),
    ("BDD_LABEL_JSON", BDD_LABEL_JSON),
    ("BDD_SEG_TRAIN",  BDD_SEG_TRAIN),
    ("BDD_SEG_MASK",   BDD_SEG_MASK),
    ("GTSRB_ROOT",     GTSRB_ROOT),
    ("VEHICLE_DIR",    VEHICLE_DIR),
]:
    mark = "✓" if _v else "✗ MISSING"
    print(f"  [{mark}]  {_n}: {_v}")

print("\n── /kaggle/input tree (depth ≤ 5) ──")
for _dp, _dirs, _files in os.walk(INPUT_ROOT):
    _depth = _dp.replace(INPUT_ROOT, "").count(os.sep)
    if _depth > 5:
        _dirs.clear(); continue
    print("  " * _depth + os.path.basename(_dp) + "/")
    if _depth == 5:
        for _f in _files[:3]:
            print("  " * (_depth + 1) + _f)


# ============================================================
#   STEP 2 — CONSTANTS & TRANSFORMS
# ============================================================

TASK_DETECT  = "detect"
TASK_SEG     = "seg"
TASK_SIGN    = "sign"
TASK_VEHICLE = "vehicle"

IMG_SIZE = (640, 384)   # (W, H)

IMG_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
AUG_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
])


# ============================================================
#   STEP 3 — DATASET CLASSES
# ============================================================

class BDD100KDetectDataset(Dataset):
    LABEL_MAP = {
        "car": 0, "bus": 1, "truck": 2, "person": 3,
        "bicycle": 4, "motorcycle": 5,
        "traffic light": 6, "traffic sign": 7, "rider": 8,
    }

    def __init__(self, img_dir, label_json, transform=None, max_samples=None):
        self.img_dir   = img_dir
        self.transform = transform or IMG_TRANSFORM
        self.samples   = []

        if not img_dir or not label_json:
            print("BDD100K detect: paths missing — skipping."); return
        if not os.path.isfile(label_json):
            print(f"BDD100K detect: JSON not a file: {label_json}"); return

        with open(label_json) as f:
            data = json.load(f)

        for item in data:
            img_path = os.path.join(img_dir, item["name"])
            if not os.path.exists(img_path): continue
            boxes = []
            for lbl in (item.get("labels") or []):
                cat = lbl.get("category", "")
                if cat in self.LABEL_MAP and "box2d" in lbl:
                    b = lbl["box2d"]
                    boxes.append([self.LABEL_MAP[cat],
                                  b["x1"], b["y1"], b["x2"], b["y2"]])
            self.samples.append({"img": img_path, "boxes": boxes})
            if max_samples and len(self.samples) >= max_samples: break

        print(f"BDD100K detect: {len(self.samples):,} samples")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        img = cv2.cvtColor(
            cv2.resize(cv2.imread(s["img"]), IMG_SIZE),
            cv2.COLOR_BGR2RGB)
        img   = self.transform(img)
        boxes = (torch.tensor(s["boxes"], dtype=torch.float32)
                 if s["boxes"] else torch.zeros((0, 5)))
        return img, {"task": TASK_DETECT, "boxes": boxes}


class BDD100KSegDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None, max_samples=None):
        self.transform = transform or IMG_TRANSFORM
        self.samples   = []

        if not mask_dir or not os.path.isdir(mask_dir):
            print("BDD100K seg: mask dir missing — skipping."); return

        img_files = {}
        if img_dir and os.path.isdir(img_dir):
            for f in os.listdir(img_dir):
                if f.lower().endswith((".jpg", ".png")):
                    img_files[Path(f).stem] = os.path.join(img_dir, f)

        for mf in sorted(os.listdir(mask_dir)):
            if not mf.lower().endswith(".png"): continue
            stem     = Path(mf).stem
            img_stem = stem.replace("_drivable_id", "").replace("_train_id", "")
            img_path = (img_files.get(img_stem)
                        or img_files.get(stem)
                        or os.path.join(img_dir or "", img_stem + ".jpg"))
            self.samples.append({"img": img_path,
                                  "mask": os.path.join(mask_dir, mf)})
            if max_samples and len(self.samples) >= max_samples: break

        print(f"BDD100K seg: {len(self.samples):,} samples")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        if s["img"] and os.path.exists(s["img"]):
            img = cv2.resize(
                cv2.cvtColor(cv2.imread(s["img"]), cv2.COLOR_BGR2RGB),
                IMG_SIZE)
        else:
            img = np.zeros((IMG_SIZE[1], IMG_SIZE[0], 3), np.uint8)
        img      = self.transform(img)
        mask_raw = np.array(
            Image.open(s["mask"]).resize(IMG_SIZE, Image.NEAREST))
        seg = torch.from_numpy(
            (mask_raw > 0).astype(np.float32)).unsqueeze(0)
        return img, {"task": TASK_SEG, "seg": seg}


SIGN_CLASSES = [
    "speed_20", "speed_30", "speed_50", "speed_60", "speed_70",
    "speed_80", "end_speed_80", "speed_100", "speed_120",
    "no_overtake", "no_overtake_trucks", "priority_road",
    "yield", "stop", "no_vehicles", "no_trucks", "no_entry",
    "general_danger", "curve_left", "curve_right", "double_curve",
    "bumpy_road", "slippery_road", "narrow_right", "narrow_left",
    "roadworks", "traffic_lights_ahead", "pedestrian", "children",
    "bicycle", "ice_snow", "wild_animals", "end_restrictions",
    "turn_right", "turn_left", "straight_only", "straight_right",
    "straight_left", "keep_right", "keep_left", "roundabout",
    "end_no_overtake", "end_no_overtake_trucks",
]
CRITICAL_SIGNS = {
    "stop": "STOP", "yield": "YIELD", "no_entry": "NO_ENTRY",
    "speed_20": 20, "speed_30": 30, "speed_50": 50,
    "speed_60": 60, "speed_70": 70, "speed_80": 80,
    "speed_100": 100, "speed_120": 120,
}


class GTSRBDataset(Dataset):
    def __init__(self, root, transform=None, max_samples=None):
        self.transform = transform or IMG_TRANSFORM
        self.samples   = []

        if not root or not os.path.exists(root):
            print("GTSRB: root missing — skipping."); return

        loaded = False

        for td in ["Train", "train"]:
            train_dir = os.path.join(root, td)
            if not os.path.isdir(train_dir): continue
            for cid in range(43):
                for fmt in [str(cid), f"{cid:05d}", f"{cid:02d}"]:
                    cdir = os.path.join(train_dir, fmt)
                    if not os.path.isdir(cdir): continue
                    for f in os.listdir(cdir):
                        if Path(f).suffix.lower() in {".png", ".ppm", ".jpg"}:
                            self.samples.append((os.path.join(cdir, f), cid))
                    break
                if max_samples and len(self.samples) >= max_samples: break
            if self.samples: loaded = True; break

        if not loaded:
            for csv_name in ["Train.csv", "train.csv"]:
                csv_path = os.path.join(root, csv_name)
                if not os.path.exists(csv_path): continue
                df = pd.read_csv(csv_path)
                for _, row in df.iterrows():
                    p   = os.path.join(root, str(row.get("Path", "")))
                    lbl = int(row.get("ClassId", 0))
                    if os.path.exists(p):
                        self.samples.append((p, lbl))
                    if max_samples and len(self.samples) >= max_samples: break
                if self.samples: loaded = True; break

        if not loaded:
            candidate = _find_dir_with_image_subfolders(root, min_classes=10)
            if candidate:
                subdirs = sorted([d for d in os.listdir(candidate)
                                  if os.path.isdir(os.path.join(candidate, d))])
                for cid, sd in enumerate(subdirs):
                    cdir = os.path.join(candidate, sd)
                    for f in os.listdir(cdir):
                        if Path(f).suffix.lower() in {".png", ".ppm", ".jpg"}:
                            self.samples.append((os.path.join(cdir, f), cid))
                    if max_samples and len(self.samples) >= max_samples: break

        print(f"GTSRB sign: {len(self.samples):,} samples, 43 classes")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = np.array(Image.open(path).convert("RGB").resize((64, 64)))
        img = self.transform(img)
        return img, {"task": TASK_SIGN,
                     "sign_label": torch.tensor(label, dtype=torch.long)}


class RoadVehicleDataset(Dataset):
    def __init__(self, root_dir, transform=None, max_samples=None):
        self.transform = transform or IMG_TRANSFORM
        self.samples   = []
        self.classes   = []

        if not root_dir or not os.path.isdir(root_dir):
            print("Road Vehicle: directory missing — skipping."); return

        subdirs    = [d for d in os.listdir(root_dir)
                      if os.path.isdir(os.path.join(root_dir, d))]
        class_dirs = [sd for sd in subdirs
                      if any(Path(f).suffix.lower() in {".jpg", ".png", ".jpeg"}
                             for f in os.listdir(os.path.join(root_dir, sd)))]

        if class_dirs:
            self.classes = sorted(class_dirs)
            print(f"Road Vehicle classes ({len(self.classes)}): {self.classes}")
            for cid, cname in enumerate(self.classes):
                cdir = os.path.join(root_dir, cname)
                for f in os.listdir(cdir):
                    if Path(f).suffix.lower() in {".jpg", ".png", ".jpeg"}:
                        self.samples.append((os.path.join(cdir, f), cid))
                if max_samples and len(self.samples) >= max_samples: break

        if not self.samples:
            print("Road Vehicle: no class-folders — scanning recursively…")
            all_imgs = []
            base_depth = root_dir.rstrip("/").count("/")
            for dp, dirs, files in os.walk(root_dir):
                if dp.count("/") - base_depth > 6:
                    dirs.clear(); continue
                for f in files:
                    if Path(f).suffix.lower() in {".jpg", ".png", ".jpeg"}:
                        all_imgs.append(os.path.join(dp, f))
                        if max_samples and len(all_imgs) >= max_samples:
                            dirs.clear(); break

            if all_imgs:
                from collections import defaultdict
                buckets = defaultdict(list)
                for p in all_imgs:
                    buckets[os.path.basename(os.path.dirname(p))].append(p)
                if len(buckets) >= 2:
                    self.classes = sorted(buckets.keys())
                    for cid, cname in enumerate(self.classes):
                        for p in buckets[cname]:
                            self.samples.append((p, cid))
                        if max_samples and len(self.samples) >= max_samples: break
                else:
                    self.classes = ["vehicle"]
                    for p in all_imgs[:max_samples or len(all_imgs)]:
                        self.samples.append((p, 0))

        if not self.samples:
            print("Road Vehicle: no images found — skipping."); return

        print(f"Road Vehicle: {len(self.samples):,} samples, "
              f"{len(self.classes)} classes")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = np.array(Image.open(path).convert("RGB").resize((128, 128)))
        img = self.transform(img)
        return img, {"task": TASK_VEHICLE,
                     "vehicle_label": torch.tensor(label, dtype=torch.long)}


# ============================================================
#   STEP 4 — TASK-MASKED DATALOADER
# ============================================================

def task_masked_collate(batch):
    imgs, metas = zip(*batch)
    task        = metas[0]["task"]
    img_batch   = torch.stack(imgs)
    if task == TASK_DETECT:
        return img_batch, {"task": task, "boxes": [m["boxes"] for m in metas]}
    if task == TASK_SEG:
        return img_batch, {"task": task,
                           "seg": torch.stack([m["seg"] for m in metas])}
    if task == TASK_SIGN:
        return img_batch, {"task": task,
                           "sign_label": torch.stack(
                               [m["sign_label"] for m in metas])}
    if task == TASK_VEHICLE:
        return img_batch, {"task": task,
                           "vehicle_label": torch.stack(
                               [m["vehicle_label"] for m in metas])}
    return img_batch, {"task": task}


class TaskSampledLoader:
    """Round-robin across per-task DataLoaders."""
    def __init__(self, loaders):
        self.loaders = loaders
        self.iters   = {k: iter(v) for k, v in loaders.items()}

    def __iter__(self):
        keys = list(self.loaders.keys())
        while True:
            random.shuffle(keys)
            for k in keys:
                try:
                    yield next(self.iters[k])
                except StopIteration:
                    self.iters[k] = iter(self.loaders[k])
                    yield next(self.iters[k])

    def __len__(self):
        return sum(len(v) for v in self.loaders.values())


# ============================================================
#   STEP 5 — MODEL
#
#   ONNX FIX 5: SegmentationHead previously used
#     F.adaptive_avg_pool2d(c4, c3.shape[-2:])
#   The TorchScript ONNX exporter requires adaptive_avg_pool2d's
#   output_size to be a compile-time constant, but c3.shape[-2:]
#   is a runtime value → SymbolicValueError.
#   Fix: use F.interpolate(..., size=c3.shape[-2:]) instead.
#   The ONNX Resize op supports dynamic target sizes at opset >= 11.
#
#   ONNX FIX 6: SignHead / VehicleClassHead previously used
#     nn.AdaptiveAvgPool2d(1)
#   Same exporter limitation. Fix: replace with .mean(dim=[2,3])
#   which maps to ONNX ReduceMean with static axes=[2,3].
# ============================================================

class ConvBnRelu(nn.Module):
    def __init__(self, cin, cout, k=3, s=1, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, k, s, p, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

    def forward(self, x): return self.block(x)


class SharedBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        base     = models.mobilenet_v3_small(
            weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        features = list(base.features)
        self.c2  = nn.Sequential(*features[:4])
        self.c3  = nn.Sequential(*features[4:9])
        self.c4  = nn.Sequential(*features[9:])

    def forward(self, x):
        c2 = self.c2(x)
        c3 = self.c3(c2)
        c4 = self.c4(c3)
        return c2, c3, c4


class DetectionHead(nn.Module):
    def __init__(self, in_ch=576, num_classes=9):
        super().__init__()
        self.neck = nn.Sequential(ConvBnRelu(in_ch, 256), ConvBnRelu(256, 256))
        self.cls  = nn.Conv2d(256, num_classes, 1)
        self.box  = nn.Conv2d(256, 4, 1)
        self.obj  = nn.Conv2d(256, 1, 1)

    def forward(self, c4):
        x = self.neck(c4)
        return self.cls(x), self.box(x), self.obj(x)


class SegmentationHead(nn.Module):
    def __init__(self, c2=16, c3=48, c4=576):
        super().__init__()
        self.lat3  = ConvBnRelu(c3, 64, 1, 1, 0)
        self.lat2  = ConvBnRelu(c2, 64, 1, 1, 0)
        self.merge = ConvBnRelu(64 + c4, 128)
        self.up1   = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            ConvBnRelu(128, 64))
        self.up2   = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            ConvBnRelu(128, 64))
        self.up3   = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            ConvBnRelu(64, 32))
        self.out   = nn.Conv2d(32, 1, 1)

    def forward(self, c2, c3, c4):
        lat3 = self.lat3(c3)
        lat2 = self.lat2(c2)
        # FIX 5: F.interpolate supports dynamic target size in ONNX Resize op.
        #        F.adaptive_avg_pool2d does NOT support dynamic output_size.
        c4_resized = F.interpolate(c4, size=(c3.shape[2], c3.shape[3]),
                                   mode="bilinear", align_corners=False)
        x = self.merge(torch.cat([c4_resized, lat3], dim=1))
        x = self.up1(x)
        x = self.up2(torch.cat([x, lat2], dim=1))
        x = self.up3(x)
        return F.interpolate(self.out(x), scale_factor=4,
                             mode="bilinear", align_corners=False)


class SignHead(nn.Module):
    def __init__(self, in_ch=576, num_classes=43):
        super().__init__()
        # FIX 6: .mean(dim=[2,3]) → ONNX ReduceMean with static axes.
        #        AdaptiveAvgPool2d(1) fails the TorchScript exporter.
        self.mlp = nn.Sequential(
            nn.Linear(in_ch, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes))

    def forward(self, c4):
        return self.mlp(c4.mean(dim=[2, 3]))


class VehicleClassHead(nn.Module):
    def __init__(self, in_ch=576, num_classes=10):
        super().__init__()
        # FIX 6: same as SignHead.
        self.mlp = nn.Sequential(
            nn.Linear(in_ch, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes))

    def forward(self, c4):
        return self.mlp(c4.mean(dim=[2, 3]))


class ADASModel(nn.Module):
    def __init__(self, num_vehicle_classes=10):
        super().__init__()
        self.backbone  = SharedBackbone()
        self.det_head  = DetectionHead(576, 9)
        self.seg_head  = SegmentationHead()
        self.sign_head = SignHead(576, 43)
        self.veh_head  = VehicleClassHead(576, num_vehicle_classes)

    def forward(self, x, task=None):
        c2, c3, c4 = self.backbone(x)
        if task == TASK_DETECT:  return self.det_head(c4)
        if task == TASK_SEG:     return self.seg_head(c2, c3, c4)
        if task == TASK_SIGN:    return self.sign_head(c4)
        if task == TASK_VEHICLE: return self.veh_head(c4)
        return {
            "det":  self.det_head(c4),
            "seg":  self.seg_head(c2, c3, c4),
            "sign": self.sign_head(c4),
            "veh":  self.veh_head(c4),
        }


# ============================================================
#   STEP 6 — MULTI-TASK LOSS
# ============================================================

class MultiTaskLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(4))
        self.bce = nn.BCEWithLogitsLoss()
        self.ce  = nn.CrossEntropyLoss()

    def _w(self, loss, i):
        lv = self.log_vars[i]
        return torch.exp(-lv) * loss + lv

    def forward(self, preds, meta):
        task = meta["task"]
        if task == TASK_DETECT:
            _, _, obj = preds
            loss = self.bce(obj, torch.zeros_like(obj))
            return self._w(loss, 0), {"det": loss.item()}
        if task == TASK_SEG:
            gt = meta["seg"].to(DEVICE)
            if preds.shape[-2:] != gt.shape[-2:]:
                preds = F.interpolate(preds, gt.shape[-2:],
                                      mode="bilinear", align_corners=False)
            loss = self.bce(preds, gt)
            return self._w(loss, 1), {"seg": loss.item()}
        if task == TASK_SIGN:
            loss = self.ce(preds, meta["sign_label"].to(DEVICE))
            return self._w(loss, 2), {"sign": loss.item()}
        if task == TASK_VEHICLE:
            loss = self.ce(preds, meta["vehicle_label"].to(DEVICE))
            return self._w(loss, 3), {"veh": loss.item()}
        return torch.tensor(0.0, requires_grad=True), {}


# ============================================================
#   STEP 7 — TRAINING LOOP
# ============================================================

def train(model, loader, epochs=30, steps_per_epoch=400,
          save_path="/kaggle/working/adas_model_best.pth"):

    criterion = MultiTaskLoss().to(DEVICE)
    optimizer = torch.optim.AdamW(
        list(model.parameters()) + list(criterion.parameters()),
        lr=2e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=2e-4,
        steps_per_epoch=steps_per_epoch, epochs=epochs)

    # FIX 3: updated GradScaler API (torch >= 2.4)
    use_amp = (DEVICE == "cuda")
    scaler  = torch.amp.GradScaler("cuda", enabled=use_amp)

    best_loss = float("inf")
    data_iter = iter(loader)
    history   = []
    print(f"\nTraining — {epochs} epochs × {steps_per_epoch} steps")
    print(f"Active tasks : {list(loader.loaders.keys())}")
    print("=" * 65)

    for epoch in range(epochs):
        model.train()
        running = 0.0
        ep_loss = {}

        for _ in range(steps_per_epoch):
            imgs, meta = next(data_iter)
            imgs = imgs.to(DEVICE)
            task = meta["task"]

            # FIX 3: updated autocast API
            with torch.amp.autocast("cuda", enabled=use_amp):
                preds      = model(imgs, task=task)
                loss, info = criterion(preds, meta)

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running += loss.item()
            for k, v in info.items():
                ep_loss[k] = ep_loss.get(k, 0) + v

        avg   = running / steps_per_epoch
        history.append(avg)
        lam   = torch.exp(-criterion.log_vars.detach().cpu()).numpy()
        lam_s = " ".join(f"{x:.2f}" for x in lam)
        tsk_s = " | ".join(
            f"{k} {v / max(steps_per_epoch // 4, 1):.4f}"
            for k, v in ep_loss.items())
        saved = ""
        if avg < best_loss:
            best_loss = avg
            torch.save({
                "epoch":        epoch + 1,
                "model":        model.state_dict(),
                "loss_weights": criterion.state_dict(),
                "optimizer":    optimizer.state_dict(),
            }, save_path)
            saved = " ✓ saved"
        print(f"Epoch {epoch+1:03d}/{epochs} | Loss {avg:.4f} | "
              f"{tsk_s} | λ [{lam_s}]{saved}")

    print(f"\nBest loss : {best_loss:.4f}\nWeights   : {save_path}")
    return history


# ============================================================
#   STEP 8 — ONNX EXPORT
#   FIX 4: dynamo=False → legacy TorchScript exporter (no onnxscript)
#   FIX 5+6 in model classes above make all ops ONNX-traceable.
# ============================================================

def export_onnx(model, path="/kaggle/working/adas_model.onnx"):
    model.eval()
    dummy = torch.randn(1, 3, IMG_SIZE[1], IMG_SIZE[0]).to(DEVICE)

    class _Wrapper(nn.Module):
        def __init__(self, m): super().__init__(); self.m = m

        def forward(self, x):
            o = self.m(x, task=None)
            cls, box, obj = o["det"]
            return cls, box, obj, o["seg"], o["sign"], o["veh"]

    torch.onnx.export(
        _Wrapper(model).to(DEVICE),
        dummy,
        path,
        opset_version=17,
        dynamo=False,           # legacy TorchScript exporter — no onnxscript
        input_names=["images"],
        output_names=["det_cls", "det_box", "det_obj", "seg", "sign", "veh"],
        dynamic_axes={n: {0: "batch"} for n in
                      ["images", "det_cls", "det_box", "det_obj",
                       "seg", "sign", "veh"]},
        do_constant_folding=True,
    )
    print(f"ONNX exported → {path}")
    return path


# ============================================================
#   STEP 9 — INFERENCE + VISUALIZATION
# ============================================================

def infer(model, img_bgr, vehicle_classes):
    model.eval()
    img_rgb = cv2.cvtColor(cv2.resize(img_bgr, IMG_SIZE), cv2.COLOR_BGR2RGB)
    x = IMG_TRANSFORM(img_rgb).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = model(x, task=None)

    seg_mask  = (torch.sigmoid(out["seg"][0, 0]).cpu().numpy() > 0.5
                 ).astype(np.uint8)
    sign_idx  = out["sign"][0].argmax().item()
    sign_name = SIGN_CLASSES[sign_idx]
    sign_conf = F.softmax(out["sign"][0], dim=0)[sign_idx].item()
    veh_idx   = out["veh"][0].argmax().item()
    veh_conf  = F.softmax(out["veh"][0], dim=0)[veh_idx].item()
    veh_name  = (vehicle_classes[veh_idx]
                 if veh_idx < len(vehicle_classes) else f"class_{veh_idx}")

    decision = "AVANCER"
    if sign_name in CRITICAL_SIGNS:
        val = CRITICAL_SIGNS[sign_name]
        if val == "STOP":          decision = "STOP"
        elif val == "YIELD":       decision = "RALENTIR"
        elif val == "NO_ENTRY":    decision = "STOP — SENS INTERDIT"
        elif isinstance(val, int): decision = f"LIMITE {val} km/h"

    return {
        "seg_mask": seg_mask,
        "sign":    (sign_name, sign_conf),
        "vehicle": (veh_name, veh_conf),
        "decision": decision,
    }


def visualize(img_bgr, result, vehicle_classes):
    img    = cv2.resize(img_bgr, IMG_SIZE).copy()
    h, w   = img.shape[:2]
    seg    = cv2.resize(result["seg_mask"], (w, h),
                        interpolation=cv2.INTER_NEAREST)
    overlay = img.copy()
    overlay[seg == 1] = (
        overlay[seg == 1] * 0.45 + np.array([0, 180, 0]) * 0.55
    ).clip(0, 255)
    img = overlay.astype(np.uint8)

    sn, sc = result["sign"]
    vn, vc = result["vehicle"]
    dec    = result["decision"]
    dc = {"AVANCER": (0, 255, 0), "STOP": (0, 0, 255),
          "RALENTIR": (0, 165, 255)}.get(dec.split()[0], (255, 255, 255))

    cv2.rectangle(img, (0, 0), (380, 95), (0, 0, 0), -1)
    cv2.putText(img, f"Decision: {dec}", (8, 26),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, dc, 2)
    cv2.putText(img, f"Sign    : {sn}  {sc:.0%}", (8, 53),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 200, 0), 1)
    cv2.putText(img, f"Vehicle : {vn}  {vc:.0%}", (8, 78),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 220, 255), 1)
    cv2.putText(img, f"Road: {int(seg.mean() * 100)}%", (w - 115, 26),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 128), 1)
    return img


# ============================================================
#   STEP 10 — MAIN
# ============================================================

if __name__ == "__main__":

    # ── 1. Build datasets ────────────────────────────────────
    print("\n── Building datasets ──")
    det_ds  = BDD100KDetectDataset(BDD_IMG_TRAIN, BDD_LABEL_JSON,
                                   AUG_TRANSFORM, max_samples=20_000)
    seg_ds  = BDD100KSegDataset(BDD_SEG_TRAIN, BDD_SEG_MASK,
                                AUG_TRANSFORM, max_samples=10_000)
    sign_ds = GTSRBDataset(GTSRB_ROOT, AUG_TRANSFORM)
    veh_ds  = RoadVehicleDataset(VEHICLE_DIR, AUG_TRANSFORM)

    num_vehicle_classes = max(len(veh_ds.classes), 1) if veh_ds.classes else 4
    vehicle_classes     = (veh_ds.classes or
                           [f"cls_{i}" for i in range(num_vehicle_classes)])

    # ── 2. Build loaders ─────────────────────────────────────
    print("\n── Building loaders ──")
    loaders = {}
    BS      = 16
    for ds, task in [(det_ds,  TASK_DETECT),
                     (seg_ds,  TASK_SEG),
                     (sign_ds, TASK_SIGN),
                     (veh_ds,  TASK_VEHICLE)]:
        if len(ds) > 0:
            loaders[task] = DataLoader(
                ds, batch_size=BS, shuffle=True,
                collate_fn=task_masked_collate,
                num_workers=2, pin_memory=True, drop_last=True)

    if not loaders:
        print("\n" + "=" * 65)
        print("❌  No datasets loaded.")
        print("    The /kaggle/input tree is printed above.")
        print("    Check that all 4 datasets are attached via '+ Add Input'.")
        print("=" * 65)
        raise RuntimeError("No datasets loaded.")

    print(f"✓ Tasks loaded: {list(loaders.keys())}")
    sampler = TaskSampledLoader(loaders)

    # ── 3. Build model ───────────────────────────────────────
    print("\n── Building model ──")
    model = ADASModel(num_vehicle_classes).to(DEVICE)
    total = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"Parameters : {total:.1f} M")
    print(f"Heads      : detect | seg | sign (43) | "
          f"vehicle ({num_vehicle_classes})")

    # ── 4. Train ─────────────────────────────────────────────
    history = train(model, sampler, epochs=30, steps_per_epoch=400)

    # ── 5. Loss curve ────────────────────────────────────────
    plt.figure(figsize=(10, 4))
    plt.plot(history, marker="o", color="royalblue", linewidth=2)
    plt.title("Multi-task training loss (det + seg + sign + vehicle)")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("/kaggle/working/loss_curve.png", dpi=150)
    plt.show()

    # ── 6. ONNX export ───────────────────────────────────────
    export_onnx(model)

    # ── 7. Inference demo ────────────────────────────────────
    print("\n── Inference demo on val images ──")
    if BDD_IMG_VAL and os.path.isdir(BDD_IMG_VAL):
        val_imgs = sorted(os.listdir(BDD_IMG_VAL))[:6]
        fig, axes = plt.subplots(2, 3, figsize=(20, 10))
        for i, name in enumerate(val_imgs):
            img = cv2.imread(os.path.join(BDD_IMG_VAL, name))
            if img is None: continue
            result = infer(model, img, vehicle_classes)
            vis    = visualize(img, result, vehicle_classes)
            axes.flat[i].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
            axes.flat[i].set_title(
                f"{result['decision']}  |  {result['sign'][0]}",
                fontsize=11)
            axes.flat[i].axis("off")
        plt.suptitle(
            "ADAS — BDD100K (det+seg) + GTSRB (sign) + Road Vehicle (cls)",
            fontsize=13)
        plt.tight_layout()
        plt.savefig("/kaggle/working/inference_demo.png", dpi=150)
        plt.show()

    print("\n✓ Done. Files saved to /kaggle/working/")